# Diccionarios · Avanzado

`get`, `setdefault` y `|`, comprensiones, invertir y agrupar, el módulo `collections` (`defaultdict`, `Counter`, `namedtuple`, `OrderedDict`, `ChainMap`), diccionarios anidados, JSON y memoización.

## 1. `get`, `setdefault` y los operadores `|` y `|=`

Con un inventario de fruta: consultar existencias con `get` (con y sin valor por defecto), registrar varios proveedores por fruta con `setdefault` y combinar inventarios con `|` (crea un diccionario nuevo) y `|=` (modifica el original, y admite también una lista de parejas). Comprobar que, si una clave se repite, gana el operando de la derecha.

In [1]:
inventario = {"manzanas": 12, "peras": 5}
print(inventario.get("plátanos"), inventario.get("plátanos", 0), inventario["peras"])

proveedores = {}
for fruta, proveedor in [("manzanas", "Finca Tejeda"), ("peras", "Agrícola Sur"), ("manzanas", "Huerta Valsequillo")]:
    proveedores.setdefault(fruta, []).append(proveedor)
print(proveedores)
print(proveedores.setdefault("peras", ["otro"]), proveedores.setdefault("kiwis", []), list(proveedores))

None 0 5
{'manzanas': ['Finca Tejeda', 'Huerta Valsequillo'], 'peras': ['Agrícola Sur']}
['Agrícola Sur'] [] ['manzanas', 'peras', 'kiwis']


In [2]:
nuevo = {"peras": 20, "kiwis": 8}
print(inventario | nuevo)
print(nuevo | inventario)
original = inventario
inventario |= nuevo
inventario |= [("uvas", 3)]
print(inventario, original is inventario)
assert {**{"a": 1}, **{"a": 2, "b": 3}} == {"a": 1} | {"a": 2, "b": 3} == {"a": 2, "b": 3}

{'manzanas': 12, 'peras': 20, 'kiwis': 8}
{'peras': 5, 'kiwis': 8, 'manzanas': 12}
{'manzanas': 12, 'peras': 20, 'kiwis': 8, 'uvas': 3} True


## 2. Comprensiones de diccionarios

Usar comprensiones para: crear los cuadrados del 1 al 10, quedarse con los productos de más de 2 €, aplicar un 7 % de IGIC a todos los precios, construir un diccionario a partir de dos listas con `zip`, clasificar números en par o impar con una expresión condicional y crear una tabla de multiplicar anidada.

In [3]:
precios = {"pan": 1.2, "leche": 0.95, "queso": 6.5, "café": 3.4}

print({n: n * n for n in range(1, 11)})
print({k: v for k, v in precios.items() if v > 2})
print({k: round(v * 1.07, 2) for k, v in precios.items()})
print(dict(zip(["lun", "mar", "mié"], [8, 6, 7])), {d: h for d, h in zip(["lun", "mar", "mié"], [8, 6, 7])})
print({n: "par" if n % 2 == 0 else "impar" for n in range(1, 6)})
tabla = {i: {j: i * j for j in range(1, 4)} for i in range(1, 4)}
print(tabla, tabla[3][2])

{1: 1, 2: 4, 3: 9, 4: 16, 5: 25, 6: 36, 7: 49, 8: 64, 9: 81, 10: 100}
{'queso': 6.5, 'café': 3.4}
{'pan': 1.28, 'leche': 1.02, 'queso': 6.96, 'café': 3.64}
{'lun': 8, 'mar': 6, 'mié': 7} {'lun': 8, 'mar': 6, 'mié': 7}
{1: 'impar', 2: 'par', 3: 'impar', 4: 'par', 5: 'impar'}
{1: {1: 1, 2: 2, 3: 3}, 2: {1: 2, 2: 4, 3: 6}, 3: {1: 3, 2: 6, 3: 9}} 6


## 3. Invertir un diccionario

Invertir un diccionario `clave → valor`. Si los valores son únicos basta una comprensión, y al invertir dos veces se recupera el original; si se repiten, la comprensión pierde claves, así que hay que agrupar las claves en listas. Escribir también una función que diga si un diccionario se puede invertir sin perder información.

In [4]:
capitales = {"España": "Madrid", "Francia": "París", "Portugal": "Lisboa"}
inverso = {v: k for k, v in capitales.items()}
print(inverso)
assert {v: k for k, v in inverso.items()} == capitales

notas = {"Ana": 9, "Luis": 7, "Eva": 9, "Raúl": 5, "Sara": 7}
print({v: k for k, v in notas.items()})

def invierte(d):
    inv = {}
    for k, v in d.items():
        inv.setdefault(v, []).append(k)
    return inv

def es_invertible(d):
    return len(set(d.values())) == len(d)

print(invierte(notas))
print(es_invertible(capitales), es_invertible(notas))
assert sorted(k for ks in invierte(notas).values() for k in ks) == sorted(notas)

{'Madrid': 'España', 'París': 'Francia', 'Lisboa': 'Portugal'}
{9: 'Eva', 7: 'Sara', 5: 'Raúl'}
{9: ['Ana', 'Eva'], 7: ['Luis', 'Sara'], 5: ['Raúl']}
True False


## 4. Agrupar con `defaultdict`

Agrupar una lista de alumnos `(nombre, grupo, nota)` por grupo de tres formas: con `setdefault`, con `defaultdict(list)` y con `itertools.groupby` (que exige ordenar antes). Calcular la nota media de cada grupo con `defaultdict(float)` y `defaultdict(int)`, y construir un árbol de categorías con un `defaultdict` recursivo. ¿Qué pasa al consultar una clave que no existe en un `defaultdict`?

In [5]:
from collections import defaultdict
from itertools import groupby

alumnos = [("Ana", "B", 8.5), ("Luis", "A", 6.0), ("Eva", "B", 9.1), ("Raúl", "C", 4.5), ("Sara", "A", 7.2), ("Iván", "B", 5.0)]

g1 = {}
for nombre, grupo, _ in alumnos:
    g1.setdefault(grupo, []).append(nombre)

g2 = defaultdict(list)
for nombre, grupo, _ in alumnos:
    g2[grupo].append(nombre)

por_grupo = lambda a: a[1]
g3 = {g: [a[0] for a in filas] for g, filas in groupby(sorted(alumnos, key=por_grupo), key=por_grupo)}

print(g1)
print(g2)
print(g3)
assert g1 == g2 == g3
print(dict(g2), g2["D"], "D" in g2)

{'B': ['Ana', 'Eva', 'Iván'], 'A': ['Luis', 'Sara'], 'C': ['Raúl']}
defaultdict(<class 'list'>, {'B': ['Ana', 'Eva', 'Iván'], 'A': ['Luis', 'Sara'], 'C': ['Raúl']})
{'A': ['Luis', 'Sara'], 'B': ['Ana', 'Eva', 'Iván'], 'C': ['Raúl']}
{'B': ['Ana', 'Eva', 'Iván'], 'A': ['Luis', 'Sara'], 'C': ['Raúl']} [] True


In [6]:
suma, cuenta = defaultdict(float), defaultdict(int)
for _, grupo, nota in alumnos:
    suma[grupo] += nota
    cuenta[grupo] += 1
print({g: round(suma[g] / cuenta[g], 2) for g in sorted(suma)})

def arbol():
    return defaultdict(arbol)

def a_dict(d):
    return {k: a_dict(v) if isinstance(v, defaultdict) else v for k, v in d.items()}

catalogo = arbol()
catalogo["frutas"]["tropicales"]["mango"] = 3.5
catalogo["frutas"]["cítricos"]["limón"] = 1.2
catalogo["verduras"]["papas"] = 0.9
print(a_dict(catalogo))

{'A': 6.6, 'B': 7.53, 'C': 4.5}
{'frutas': {'tropicales': {'mango': 3.5}, 'cítricos': {'limón': 1.2}}, 'verduras': {'papas': 0.9}}


## 5. `Counter` a fondo

Contar las letras de dos palabras con `Counter` y usar `most_common`, `total`, `elements`, `subtract` y la aritmética de contadores (`+`, `-`, `&`, `|`). Aplicarlo para obtener las palabras más frecuentes de un texto, decir si dos frases son anagramas y si se puede escribir una nota con las letras recortadas de otra.

In [7]:
from collections import Counter

a, b = Counter("abracadabra"), Counter("alacazam")
print(a, a.most_common(2), a.total(), a["z"])
print("suma:  ", a + b)
print("resta: ", a - b)
print("mínimo:", a & b)
print("máximo:", a | b)
c = a.copy()
c.subtract(b)
print("subtract (admite negativos):", c)
print("".join(Counter(a=2, b=1, c=0).elements()))

Counter({'a': 5, 'b': 2, 'r': 2, 'c': 1, 'd': 1}) [('a', 5), ('b', 2)] 11 0
suma:   Counter({'a': 9, 'b': 2, 'r': 2, 'c': 2, 'd': 1, 'l': 1, 'z': 1, 'm': 1})
resta:  Counter({'b': 2, 'r': 2, 'a': 1, 'd': 1})
mínimo: Counter({'a': 4, 'c': 1})
máximo: Counter({'a': 5, 'b': 2, 'r': 2, 'c': 1, 'd': 1, 'l': 1, 'z': 1, 'm': 1})
subtract (admite negativos): Counter({'b': 2, 'r': 2, 'a': 1, 'd': 1, 'c': 0, 'l': -1, 'z': -1, 'm': -1})
aab


In [8]:
texto = "el que mucho abarca poco aprieta y el que poco abarca mucho aprieta menos"
print(Counter(texto.split()).most_common(3))

def letras(s):
    return Counter(ch for ch in s.lower() if ch.isalpha())

def anagramas(x, y):
    return letras(x) == letras(y)

def se_puede_escribir(nota, recortes):
    return not letras(nota) - letras(recortes)

print(anagramas("Roma", "amor"), anagramas("Listen", "Silent"), anagramas("casa", "saco"))
print(se_puede_escribir("arroz", "la zorra"), se_puede_escribir("carroza", "la zorra"))
for nota, recortes in [("hola", "ahol"), ("aa", "a"), ("", "x"), ("arroz", "la zorra")]:
    assert se_puede_escribir(nota, recortes) == (letras(nota) <= letras(recortes))

[('el', 2), ('que', 2), ('mucho', 2)]
True True False
True False


## 6. `namedtuple`

Definir `Alumno = namedtuple('Alumno', 'nombre grupo nota')` y un `Punto` con valor por defecto para `y`. Acceder a los campos por nombre y por posición, desempaquetar, convertir a diccionario con `_asdict`, crear copias modificadas con `_replace` y comprobar que no se pueden modificar. Usarlos como claves de diccionario y para construir un diccionario `nombre → nota`.

In [9]:
from collections import namedtuple

Alumno = namedtuple("Alumno", "nombre grupo nota")
Punto = namedtuple("Punto", ["x", "y"], defaults=[0])

ana = Alumno("Ana", "B", 8.5)
nombre, grupo, nota = ana
print(ana, ana.nombre, ana[2], Alumno._fields, nombre)
print(ana._asdict(), ana._replace(nota=9.0), ana)
try:
    ana.nota = 10
except AttributeError as e:
    print("AttributeError:", e)

print(Punto(3), Punto._make([1, 2]), Punto(1, 2) == (1, 2))
lugares = {Punto(0, 0): "origen", Punto(1, 0): "este"}
print(lugares[Punto(1)])

clase = [Alumno(*t) for t in [("Luis", "A", 6.0), ("Eva", "B", 9.1), ("Sara", "A", 7.2)]]
print(max(clase, key=lambda al: al.nota).nombre, {al.nombre: al.nota for al in clase})

Alumno(nombre='Ana', grupo='B', nota=8.5) Ana 8.5 ('nombre', 'grupo', 'nota') Ana
{'nombre': 'Ana', 'grupo': 'B', 'nota': 8.5} Alumno(nombre='Ana', grupo='B', nota=9.0) Alumno(nombre='Ana', grupo='B', nota=8.5)
AttributeError: can't set attribute
Punto(x=3, y=0) Punto(x=1, y=2) True
este
Eva {'Luis': 6.0, 'Eva': 9.1, 'Sara': 7.2}


## 7. `OrderedDict`: caché LRU

Implementar una caché LRU (*least recently used*) de capacidad fija con `OrderedDict`: al consultar o guardar una clave se mueve al final (`move_to_end`) y, si se supera la capacidad, se elimina la usada hace más tiempo (`popitem(last=False)`). Comprobarla contra una simulación con listas. ¿En qué se diferencia `OrderedDict` de `dict` al comparar?

In [10]:
from collections import OrderedDict

class CacheLRU:
    def __init__(self, capacidad):
        self.capacidad = capacidad
        self.datos = OrderedDict()

    def get(self, clave):
        if clave not in self.datos:
            return None
        self.datos.move_to_end(clave)
        return self.datos[clave]

    def put(self, clave, valor):
        self.datos[clave] = valor
        self.datos.move_to_end(clave)
        if len(self.datos) > self.capacidad:
            self.datos.popitem(last=False)

cache = CacheLRU(3)
for k in "abc":
    cache.put(k, k.upper())
cache.get("a")
cache.put("d", "D")
print(list(cache.datos), cache.get("b"), cache.get("a"))

['c', 'a', 'd'] None A


In [11]:
import random

random.seed(0)
cache, orden, valores = CacheLRU(4), [], {}
for _ in range(2000):
    k = random.randrange(8)
    if k in orden:
        orden.remove(k)
    if random.random() < 0.5:
        esperado = valores[k] if k in valores else None
        if esperado is not None:
            orden.append(k)
        assert cache.get(k) == esperado
    else:
        valores[k] = random.random()
        orden.append(k)
        cache.put(k, valores[k])
    if len(orden) > 4:
        del valores[orden.pop(0)]
    assert list(cache.datos) == orden
print("Coincide con la simulación con listas")

d1, d2 = {"a": 1, "b": 2}, {"b": 2, "a": 1}
o1, o2 = OrderedDict(d1), OrderedDict(d2)
print(d1 == d2, o1 == o2, o1 == d2)

Coincide con la simulación con listas
True False True


## 8. `ChainMap`: configuración por capas

Combinar la configuración por defecto, la de un fichero y la de la línea de órdenes con `ChainMap`, de modo que cada clave se busque de la capa más prioritaria a la menos. Comprobar que equivale a unir las capas con `|`, que las escrituras solo afectan a la primera capa, que los cambios en las capas se ven al momento y usar `new_child` para un ámbito temporal.

In [12]:
from collections import ChainMap

por_defecto = {"tema": "claro", "idioma": "es", "tamaño": 12}
fichero = {"tema": "oscuro"}
ordenes = {"tamaño": 14}
config = ChainMap(ordenes, fichero, por_defecto)
print(config["tema"], config["idioma"], config["tamaño"], len(config), dict(config))
assert dict(config) == por_defecto | fichero | ordenes

config["idioma"] = "en"
print(ordenes, por_defecto["idioma"])
temporal = config.new_child({"tema": "alto contraste"})
print(temporal["tema"], config["tema"], temporal.parents["tema"])
fichero["tema"] = "sepia"
print(config["tema"], temporal["tema"])

oscuro es 14 3 {'tema': 'oscuro', 'idioma': 'es', 'tamaño': 14}
{'tamaño': 14, 'idioma': 'en'} es
alto contraste oscuro oscuro
sepia alto contraste


## 9. Recorrer y aplanar diccionarios anidados

Dado un diccionario anidado con la configuración de una aplicación, escribir funciones recursivas para calcular su profundidad, obtener un valor a partir de una ruta como `'bd.conexion.puerto'` (o un valor por defecto si no existe), aplanarlo en un diccionario de un solo nivel con claves separadas por puntos y reconstruirlo a partir del aplanado.

In [13]:
config = {
    "app": {"nombre": "Tienda", "debug": False},
    "bd": {"motor": "postgres", "conexion": {"host": "localhost", "puerto": 5432}},
    "idiomas": ["es", "en"],
}

def profundidad(d):
    if not isinstance(d, dict) or not d:
        return 0
    return 1 + max(profundidad(v) for v in d.values())

def obtener(d, ruta, defecto=None):
    for clave in ruta.split("."):
        if not isinstance(d, dict) or clave not in d:
            return defecto
        d = d[clave]
    return d

def aplanar(d, prefijo=""):
    plano = {}
    for k, v in d.items():
        if isinstance(v, dict) and v:
            plano |= aplanar(v, f"{prefijo}{k}.")
        else:
            plano[f"{prefijo}{k}"] = v
    return plano

def desaplanar(plano):
    d = {}
    for ruta, v in plano.items():
        *padres, ultima = ruta.split(".")
        nodo = d
        for p in padres:
            nodo = nodo.setdefault(p, {})
        nodo[ultima] = v
    return d

print(profundidad(config), obtener(config, "bd.conexion.puerto"), obtener(config, "bd.usuario", "?"), obtener(config, "app.nombre.x"))
plano = aplanar(config)
for k, v in plano.items():
    print(f"{k:20} {v!r}")
assert desaplanar(plano) == config

3 5432 ? None
app.nombre           'Tienda'
app.debug            False
bd.motor             'postgres'
bd.conexion.host     'localhost'
bd.conexion.puerto   5432
idiomas              ['es', 'en']


## 10. Fusión profunda

Escribir `fusiona(a, b)`, que combine dos diccionarios anidados sin modificarlos ni compartir objetos con ellos: si una clave está en los dos y ambos valores son diccionarios, se fusionan recursivamente; si no, gana el valor de `b`. Compararla con `a | b`, que solo fusiona el primer nivel.

In [14]:
import copy

def fusiona(a, b):
    res = copy.deepcopy(a)
    for k, v in b.items():
        if isinstance(res.get(k), dict) and isinstance(v, dict):
            res[k] = fusiona(res[k], v)
        else:
            res[k] = copy.deepcopy(v)
    return res

base = {"bd": {"host": "localhost", "puerto": 5432, "opciones": {"ssl": False, "timeout": 5}}, "debug": True}
produccion = {"bd": {"host": "10.0.0.5", "opciones": {"ssl": True}}, "debug": False, "replicas": 3}
copia = copy.deepcopy(base)

print(base | produccion)
fusion = fusiona(base, produccion)
print(fusion)
assert base == copia
assert fusion["bd"]["puerto"] == 5432 and fusion["bd"]["opciones"] == {"ssl": True, "timeout": 5}
fusion["bd"]["opciones"]["timeout"] = 30
print(base["bd"]["opciones"], produccion["bd"]["opciones"])

{'bd': {'host': '10.0.0.5', 'opciones': {'ssl': True}}, 'debug': False, 'replicas': 3}
{'bd': {'host': '10.0.0.5', 'puerto': 5432, 'opciones': {'ssl': True, 'timeout': 5}}, 'debug': False, 'replicas': 3}
{'ssl': False, 'timeout': 5} {'ssl': True}


## 11. Diccionarios y JSON

Convertir un pedido a JSON con `json.dumps` (con `indent` y `ensure_ascii=False`) y de vuelta con `json.loads`, y comprobar qué cambia en el viaje de ida y vuelta: claves numéricas, tuplas, `None` y `True`. Serializar después tipos que JSON no admite (`set`, `date`) con el parámetro `default`, y recuperarlos con `object_hook`.

In [15]:
import json

pedido = {"id": 17, "cliente": "Ana Pérez", "pagado": True, "cupón": None, "líneas": [("pan", 2), ("café", 1)], 3: "clave numérica"}
texto = json.dumps(pedido, ensure_ascii=False, indent=2)
print(texto)
vuelta = json.loads(texto)
print(vuelta == pedido, vuelta["líneas"], list(vuelta)[-1])
print(json.dumps({"ñ": 1}), json.dumps({"ñ": 1}, ensure_ascii=False), json.dumps({"b": 1, "a": 2}, sort_keys=True))

{
  "id": 17,
  "cliente": "Ana Pérez",
  "pagado": true,
  "cupón": null,
  "líneas": [
    [
      "pan",
      2
    ],
    [
      "café",
      1
    ]
  ],
  "3": "clave numérica"
}
False [['pan', 2], ['café', 1]] 3
{"\u00f1": 1} {"ñ": 1} {"a": 2, "b": 1}


In [16]:
from datetime import date

datos = {"etiquetas": {"oferta"}, "fecha": date(2026, 11, 6)}
try:
    json.dumps(datos)
except TypeError as e:
    print("TypeError:", e)

def a_json(obj):
    if isinstance(obj, set):
        return sorted(obj)
    if isinstance(obj, date):
        return obj.isoformat()
    raise TypeError(f"{type(obj).__name__} no es serializable")

def desde_json(d):
    if "fecha" in d:
        d["fecha"] = date.fromisoformat(d["fecha"])
    return d

texto = json.dumps(datos, default=a_json)
print(texto)
print(json.loads(texto, object_hook=desde_json))

TypeError: Object of type set is not JSON serializable
{"etiquetas": ["oferta"], "fecha": "2026-11-06"}
{'etiquetas': ['oferta'], 'fecha': datetime.date(2026, 11, 6)}


## 12. Memoización con diccionarios

Acelerar funciones recursivas guardando en un diccionario los resultados ya calculados. Contar las llamadas de Fibonacci con y sin memoización, escribir un decorador `memoiza` genérico y aplicarlo a contar los caminos en una cuadrícula moviéndose solo a la derecha o hacia abajo. Comprobar el resultado con `functools.cache` y con la fórmula $\binom{f + c - 2}{f - 1}$.

In [17]:
llamadas = 0

def fib_lento(n):
    global llamadas
    llamadas += 1
    return n if n < 2 else fib_lento(n - 1) + fib_lento(n - 2)

memo = {0: 0, 1: 1}

def fib_memo(n):
    global llamadas
    llamadas += 1
    if n not in memo:
        memo[n] = fib_memo(n - 1) + fib_memo(n - 2)
    return memo[n]

for f in (fib_lento, fib_memo):
    llamadas = 0
    print(f"{f.__name__:9} fib(25) = {f(25)}, llamadas: {llamadas}")

fib_lento fib(25) = 75025, llamadas: 242785
fib_memo  fib(25) = 75025, llamadas: 49


In [18]:
from functools import cache
from math import comb

def memoiza(f):
    memo = {}
    def envoltura(*args):
        if args not in memo:
            memo[args] = f(*args)
        return memo[args]
    envoltura.memo = memo
    return envoltura

@memoiza
def caminos(filas, columnas):
    if filas == 1 or columnas == 1:
        return 1
    return caminos(filas - 1, columnas) + caminos(filas, columnas - 1)

@cache
def caminos_cache(filas, columnas):
    if filas == 1 or columnas == 1:
        return 1
    return caminos_cache(filas - 1, columnas) + caminos_cache(filas, columnas - 1)

print(caminos(18, 18), caminos_cache(18, 18), len(caminos.memo))
print(caminos_cache.cache_info())
assert all(caminos(f, c) == caminos_cache(f, c) == comb(f + c - 2, f - 1) for f in range(1, 20) for c in range(1, 20))

2333606220 2333606220 323
CacheInfo(hits=256, misses=323, maxsize=None, currsize=323)
